# Module 38 — Long-Running Autonomous Agents

## Colab engineering lab
Build a durable worker that survives crashes, duplicate delivery, uncertain external effects, stale ownership, approval expiry and provider outages. The notebook uses a deterministic in-memory reference implementation so every failure is reproducible.

**Learning loop:** Predict → Build → Observe → Break → Debug → Measure → Improve → Defend.

## 0. Environment bootstrap

Run from the repository checkout for normal course use. In standalone Colab, the fallback downloads the canonical reference implementation so the lab remains executable.

In [ ]:
import sys
from pathlib import Path
MODULE = Path.cwd()
if (MODULE / '38-long-running-autonomous-agents' / 'app' / 'worker.py').exists():
    MODULE = MODULE / '38-long-running-autonomous-agents'
if not (MODULE / 'app' / 'worker.py').exists():
    import urllib.request
    MODULE = Path('/content/m38_module')
    (MODULE / 'app').mkdir(parents=True, exist_ok=True)
    raw='https://raw.githubusercontent.com/dataaispark-spec/Practical-Agentic-AI-and-RAG-Course/main/38-long-running-autonomous-agents/app/worker.py'
    urllib.request.urlretrieve(raw, MODULE / 'app' / 'worker.py')
    (MODULE / 'app' / '__init__.py').write_text('')
sys.path.insert(0, str(MODULE))
from app.worker import Budget, DurableJob, DurableStore, DurableWorker, EffectRecord, Status, retry_amplification
print('reference implementation:', MODULE / 'app' / 'worker.py')

## 1. Production mental model

```text
Goal → Task → Run → Attempt → Checkpoint
                         ↓
                   Policy + Budget
                         ↓
                  Lease / Fencing
                         ↓
                   Action Gateway
                         ↓
                    Effect Ledger
                         ↓
                 External System
                         ↓
                 Verify / Reconcile
```

The model may propose an action; the durable control plane decides whether the action is still authorized and safe.

In [ ]:
job = DurableJob('demo-001', 'reconcile settlement exceptions', total_steps=5, tenant='bank-a')
print(job.job_id, job.tenant, job.status, job.checkpoint, job.state_version)

## Lab 1 — Crash after an external effect

**Predict:** if the worker crashes after effect #1 but before checkpointing, what should happen on restart?

**Invariant:** the external effect must not be duplicated.

In [ ]:
calls=[]
worker=DurableWorker(lambda effect_id: calls.append(effect_id) or 'ok:'+effect_id)
job=DurableJob('crash-1','reconcile',total_steps=3)
worker.run(job,steps=3,crash_after_effect_at=1)
print('after crash:', job.status, job.checkpoint, calls)
job.status=Status.CREATED
worker.run(job,steps=3)
print('after recovery:', job.status, job.checkpoint, calls)
assert calls == ['crash-1:effect:0','crash-1:effect:1','crash-1:effect:2']
assert job.status is Status.COMPLETED

## Lab 2 — Delivery vs execution vs business semantics

Separate delivery, execution and business-effect semantics. At-least-once delivery plus a stable effect identity and reconciliation can provide effectively-once business semantics even when a network timeout creates uncertainty.

In [ ]:
print('retry amplification:', retry_amplification(10, retries=2, fanout=2))
assert retry_amplification(10,2,2) == 70

## Lab 3 — Timeout-after-write reconciliation

Model the external system as having applied the effect even though the worker received no response. Recovery must consult/reconcile the effect record instead of issuing a second business operation.

In [ ]:
calls=[]
worker=DurableWorker(lambda eid: calls.append(eid) or 'new-write')
job=DurableJob('timeout-1','correct ledger',total_steps=1)
job.effects['timeout-1:effect:0']=EffectRecord('timeout-1:effect:0',status='unknown')
worker.reconcile(job,'timeout-1:effect:0','confirmed-by-ledger')
job.status=Status.CREATED
worker.run(job,steps=1)
print(job.effects['timeout-1:effect:0'], calls)
assert calls == []
assert 'timeout-1:effect:0' in job.completed_effect_ids

## Lab 4 — Lease, heartbeat and fencing

**Attack:** worker A pauses; its lease expires; worker B takes ownership with a higher fence; A resumes. The authoritative store, not A's local memory, must reject the stale write.

In [ ]:
store=DurableStore(); job=DurableJob('fence-1','SOC investigation'); store.put(job)
a=store.acquire_lease('fence-1','A',now=0,ttl=2)
b=store.acquire_lease('fence-1','B',now=3,ttl=2)
try:
    store.commit_checkpoint('fence-1','A',a.fence,now=3,checkpoint=1)
    raise AssertionError('stale worker was accepted')
except RuntimeError as exc:
    print('correctly rejected:', exc)
assert b.fence > a.fence

## Lab 5 — Durable waiting

A human approval, webhook, timer or customer response should become a durable WAITING state. Do not keep a worker process asleep for hours.

In [ ]:
worker=DurableWorker(lambda eid:eid); job=DurableJob('wait-1','disable account',total_steps=1)
worker.wait(job,'analyst approval')
assert job.status is Status.WAITING and job.wait_reason=='analyst approval'
worker.resume(job)
assert job.status is Status.CREATED
print('wake contract:', job.status, job.wait_reason)

## Lab 6 — Approval and policy revalidation

Bind high-impact approval to the exact action representation and policy version. A restart is an authorization boundary.

In [ ]:
worker=DurableWorker(lambda eid:eid); job=DurableJob('approval-1','disable account')
worker.bind_approval(job,'action-hash-v1','policy-7')
assert worker.authorize(job,'action-hash-v1','policy-7')
assert not worker.authorize(job,'action-hash-v2','policy-7')
assert not worker.authorize(job,'action-hash-v1','policy-8')
print('old approval cannot authorize changed action/policy')

## Lab 7 — Cumulative budgets

Budgets must survive retries and restarts. Track steps, tool calls and cost as a monotonic ledger.

In [ ]:
budget=Budget(max_steps=2,max_tool_calls=2,max_cost=0.20)
budget.spend(steps=1,tool_calls=1,cost=0.10)
print(budget)
try:
    budget.spend(steps=1,tool_calls=1,cost=0.1001)
    raise AssertionError('budget bypass')
except RuntimeError as exc:
    print('correctly bounded:', exc)

## Lab 8 — Checkpoint strategy experiment

Compare every-step checkpointing, every-N checkpointing and event-log replay. Record write amplification, maximum replay distance and recovery time. Do not optimize only for throughput; include business risk.

In [ ]:
N=1000
for n in [1,10,100]:
    print({'checkpoint_every':n,'checkpoint_writes':(N+n-1)//n,'worst_case_replay':n-1})

## Lab 9 — Long-horizon migration

Design phases: precheck → backup → change → verify → rollback. Each phase needs preconditions, postconditions, an effect identity and a recovery rule. Test crash at every boundary.

## Lab 10 — Provider outage and retry storm

Simulate an outage and compare immediate retry, exponential backoff, pause/queue and escalation. Track retry amplification and backlog recovery. The control plane should reduce pressure on the failed dependency rather than multiply it.

## Lab 11 — Poisoned durable state

Tamper with tenant, state version, policy version or next-step data. Resume must validate schema, provenance, tenant and authority before any effect. Treat stored model content as untrusted input.

## Lab 12 — Cancellation race

Classify an effect as not-started, in-flight/uncertain, committed/reversible or committed/irreversible. Define cancellation behavior for each class; cancellation is not automatically rollback.

## Lab 13 — Backpressure and DLQ

Inject 10,000 events. Add concurrency caps, per-tenant fairness, bounded retries, queue-age alerts and a DLQ with reason codes and replay ownership. Measure backlog age rather than only queue length.

## Lab 14 — Incident reconstruction

Given task/run/attempt/effect/fence/policy identifiers, reconstruct: what the worker believed, what the authoritative store knew, which effects were applied, which outcomes were uncertain and why the recovery decision was safe.

## Lab 15 — Chaos day

Combine duplicate delivery + crash-after-effect + lease loss + approval expiry + provider outage + cancellation race + poisoned checkpoint.

**Required evidence:** failure matrix, trace timeline, effect ledger, recovery proof, benchmark metrics and residual-risk statement.

## Production design challenge — L7

Design a multi-region durable-agent platform for millions of tasks. Defend queue semantics, storage consistency, fencing, idempotency, event-driven waiting, retries/DLQ, cumulative budgets, tenant isolation, approval revalidation, observability, RPO/RTO and failover.

Explicitly state which guarantees cannot be made universally.

## Mastery gate

Pass only if the evidence demonstrates: safe restart, no duplicate protected effects, timeout-after-write reconciliation, stale-worker rejection, authority revalidation, cumulative budget enforcement, durable waiting, explicit cancellation semantics and reconstructable history.